# Clase 10 — Feature Engineering
## Dataset: `viviendas_features.csv`

Este notebook cubre el **Live Coding** de la clase (Pasos 1–11), más las
demostraciones de escalamiento, encoding, binning, transformación log y
Polynomial Features usando este mismo dataset.

> Todo el código está comentado a propósito. Ve descomentando celda por
> celda conforme avances en la presentación.

## Paso 0 — Configuración inicial

In [1]:
import pandas as pd
import numpy as np


## Paso 1 — Cargar los datos

In [26]:
df = pd.read_csv("../data/viviendas_features.csv")
print(df.head())
df.info()


   metros  habitaciones  banios  antiguedad     zona tipo_vivienda  \
0    70.6             2       1        15.6      Sur  Departamento   
1    75.6             3       1         0.6    Norte          Loft   
2   208.3             6       1         2.2  Oriente       Estudio   
3   182.1             4       3        16.4      Sur  Departamento   
4    93.5             6       1         8.5   Centro       Estudio   

  fecha_publicacion   precio  
0        2024-11-26  1036584  
1        2024-10-21  1542751  
2        2024-03-04  2353332  
3        2026-07-28  2135270  
4        2024-08-22  2076688  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   metros             600 non-null    float64
 1   habitaciones       600 non-null    int64  
 2   banios             600 non-null    int64  
 3   antiguedad         600 non-null    float64
 

## Demo — Escalas muy diferentes
Comparemos las escalas de dos columnas numéricas, tal como en la lámina
"Escalas diferentes".

In [50]:
print(df[["metros", "antiguedad"]].describe())


           metros  antiguedad
count  600.000000  600.000000
mean   107.497167    8.886833
std     40.687016    8.817713
min     28.000000    0.000000
25%     80.775000    2.675000
50%    106.650000    6.050000
75%    134.350000   12.300000
max    226.600000   45.000000


## Demo — StandardScaler
Estandarización: z = (x - mu) / sigma

In [73]:
from sklearn.preprocessing import StandardScaler

scaler_demo = StandardScaler()
metros_estandarizado = scaler_demo.fit_transform(df[["metros"]])
print(metros_estandarizado[:5])


[[-0.90761025]
 [-0.78461839]
 [ 2.47958565]
 [ 1.83510829]
 [-0.34430752]]


## Demo — MinMaxScaler
Normalización a [0, 1]

In [95]:
from sklearn.preprocessing import MinMaxScaler

minmax_demo = MinMaxScaler()
metros_normalizado = minmax_demo.fit_transform(df[["metros"]])
print(metros_normalizado[:5])


[[0.21450151]
 [0.23967774]
 [0.90785498]
 [0.77593152]
 [0.32980866]]


## Demo — Una mala idea: codificar zona como 1, 2, 3
Vamos a ver por qué esto sugiere un orden que no existe.

In [116]:
mapa_incorrecto = {"Centro": 1, "Norte": 2, "Sur": 3, "Oriente": 4}
zona_mal_codificada = df["zona"].map(mapa_incorrecto)
print(zona_mal_codificada.head())


0    3
1    2
2    4
3    3
4    1
Name: zona, dtype: int64


## Demo — One-Hot Encoding con Pandas

In [136]:
df_encoded_demo = pd.get_dummies(
    df,
    columns=["zona"],
    drop_first=True
)
print(df_encoded_demo.head())


   metros  habitaciones  banios  antiguedad tipo_vivienda fecha_publicacion  \
0    70.6             2       1        15.6  Departamento        2024-11-26   
1    75.6             3       1         0.6          Loft        2024-10-21   
2   208.3             6       1         2.2       Estudio        2024-03-04   
3   182.1             4       3        16.4  Departamento        2026-07-28   
4    93.5             6       1         8.5       Estudio        2024-08-22   

    precio  zona_Norte  zona_Oriente  zona_Sur  
0  1036584       False         False      True  
1  1542751        True         False     False  
2  2353332       False          True     False  
3  2135270       False         False      True  
4  2076688       False         False     False  


## Demo — One-Hot Encoding con Scikit-learn
(usando train/test para mostrar `handle_unknown="ignore"`)

In [155]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

X_demo_train, X_demo_test = train_test_split(df, test_size=0.2, random_state=42)

encoder_demo = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder_demo.fit(X_demo_train[["zona"]])

zona_train_encoded = encoder_demo.transform(X_demo_train[["zona"]])
zona_test_encoded = encoder_demo.transform(X_demo_test[["zona"]])

print(zona_train_encoded[:5])


[[0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]]


## Demo — Ordinal Encoding (ejemplo conceptual)
`zona` y `tipo_vivienda` son nominales (sin orden). Para ilustrar el caso
ordinal, usamos un ejemplo pequeño con una categoría de calidad.

In [173]:
from sklearn.preprocessing import OrdinalEncoder

calidad_ejemplo = pd.DataFrame({"calidad": ["Bajo", "Alto", "Medio", "Bajo", "Alto"]})

ordinal_encoder_demo = OrdinalEncoder(categories=[["Bajo", "Medio", "Alto"]])
calidad_codificada = ordinal_encoder_demo.fit_transform(calidad_ejemplo)
print(calidad_codificada)


[[0.]
 [2.]
 [1.]
 [0.]
 [2.]]


## Demo — Binning
Convertimos `antiguedad` en grupos interpretables.

In [190]:
df["grupo_antiguedad"] = pd.cut(
    df["antiguedad"],
    bins=[-0.1, 5, 15, 45],
    labels=["Nueva", "Media", "Antigua"]
)
print(df[["antiguedad", "grupo_antiguedad"]].head(10))


   antiguedad grupo_antiguedad
0        15.6          Antigua
1         0.6            Nueva
2         2.2            Nueva
3        16.4          Antigua
4         8.5            Media
5         6.9            Media
6        18.3          Antigua
7         1.6            Nueva
8         3.0            Nueva
9         4.5            Nueva


## Demo — Transformación logarítmica
`precio` está sesgado hacia la derecha; probemos log1p.

In [206]:
df["precio_log"] = np.log1p(df["precio"])
print(df[["precio", "precio_log"]].describe())


             precio  precio_log
count  6.000000e+02  600.000000
mean   1.935957e+06   14.414259
std    6.308424e+05    0.372932
min    3.500000e+05   12.765691
25%    1.498350e+06   14.219875
50%    1.906624e+06   14.460845
75%    2.391286e+06   14.687342
max    4.540744e+06   15.328602


In [ ]:
df["metros_por_habitacion"] = (
    df["metros"] 
    / 
    df["habitaciones"]
)

df["banios_por_habitacion"] = (
    df["banios"] 
    / 
    df["habitaciones"]
)

print(
    "Filas con habitaciones = 0:", 
    (df["habitaciones"] == 0).sum()
)

print(
    
    "Filas con banios < 0:", 
    (df["banios"] < 0).sum()
)

print(
    "Filas con metros <= 0:", 
    (df["metros"] <= 0).sum()
)

## Demo — Polynomial Features
Usando `metros` y `antiguedad`, igual que en la lámina.

In [221]:
from sklearn.preprocessing import PolynomialFeatures

poly_demo = PolynomialFeatures(degree=2, include_bias=False)
X_poly_demo = poly_demo.fit_transform(df[["metros", "antiguedad"]])

print(poly_demo.get_feature_names_out(["metros", "antiguedad"]))
print(X_poly_demo[:5])


['metros' 'antiguedad' 'metros^2' 'metros antiguedad' 'antiguedad^2']
[[7.060000e+01 1.560000e+01 4.984360e+03 1.101360e+03 2.433600e+02]
 [7.560000e+01 6.000000e-01 5.715360e+03 4.536000e+01 3.600000e-01]
 [2.083000e+02 2.200000e+00 4.338889e+04 4.582600e+02 4.840000e+00]
 [1.821000e+02 1.640000e+01 3.316041e+04 2.986440e+03 2.689600e+02]
 [9.350000e+01 8.500000e+00 8.742250e+03 7.947500e+02 7.225000e+01]]


---
# Live Coding — Modelo A vs Modelo B
Ahora sí, seguimos los pasos exactos de la presentación.

## Paso 2 — Limpiar fecha

In [235]:
df["fecha_publicacion"] = pd.to_datetime(df["fecha_publicacion"])


## Paso 3 — Variables temporales

In [248]:
df["mes_publicacion"] = df["fecha_publicacion"].dt.month
df["dia_semana"] = df["fecha_publicacion"].dt.dayofweek


## Paso 4 — Nuevas features (ratios)

In [260]:
df["metros_por_habitacion"] = df["metros"] / df["habitaciones"]
df["banios_por_habitacion"] = df["banios"] / df["habitaciones"]


## Validar antes de dividir
Este dataset tiene, a propósito, un par de filas con `habitaciones = 0`
para forzar la discusión de división entre cero.

In [271]:
print("Filas con habitaciones = 0:", (df["habitaciones"] == 0).sum())
print("Filas con banios < 0:", (df["banios"] < 0).sum())
print("Filas con metros <= 0:", (df["metros"] <= 0).sum())

# # decisión de negocio: eliminamos esas filas antes de modelar
df = df[df["habitaciones"] > 0].copy()


Filas con habitaciones = 0: 2
Filas con banios < 0: 0
Filas con metros <= 0: 0


## Paso 5 — X e y

In [281]:
X = df.drop(columns=["precio", "fecha_publicacion", "grupo_antiguedad", "precio_log"], errors="ignore")
y = df["precio"]


## Paso 6 — Train / Test

In [290]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)


## Paso 7 — Definir columnas numéricas y categóricas

In [298]:
numericas = [
    "metros",
    "habitaciones",
    "banios",
    "antiguedad",
    "mes_publicacion",
    "dia_semana",
    "metros_por_habitacion",
    "banios_por_habitacion"
]

categoricas = [
    "zona",
    "tipo_vivienda"
]


## Paso 8 — ColumnTransformer

In [305]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocesador = ColumnTransformer(
    transformers=[
        ("numericas", StandardScaler(), numericas),
        ("categoricas", OneHotEncoder(handle_unknown="ignore"), categoricas)
    ]
)


## Paso 9 — Pipeline (Modelo B, con Feature Engineering)

In [311]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

pipeline = Pipeline(
    steps=[
        ("prep", preprocesador),
        ("modelo", LinearRegression())
    ]
)

pipeline.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numericas', ...), ('categoricas', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfor

## Paso 10 — Predecir

In [316]:
pred = pipeline.predict(X_test)


## Paso 11 — Evaluar Modelo B

In [320]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae_b = mean_absolute_error(y_test, pred)
rmse_b = mean_squared_error(y_test, pred) ** 0.5
r2_b = r2_score(y_test, pred)

print("Modelo B -> MAE:", mae_b, " RMSE:", rmse_b, " R2:", r2_b)


Modelo B -> MAE: 89889.43759816857  RMSE: 108444.96720760677  R2: 0.96988292774666


---
## Antes vs. después — Modelo A (básico) vs Modelo B (Feature Engineering)
Modelo A usa solamente las variables numéricas originales, sin zona,
tipo_vivienda ni las variables creadas.

In [323]:
columnas_basicas = ["metros", "habitaciones", "banios", "antiguedad"]

X_train_basico = X_train[columnas_basicas]
X_test_basico = X_test[columnas_basicas]

modelo_a = LinearRegression()
modelo_a.fit(X_train_basico, y_train)
pred_a = modelo_a.predict(X_test_basico)

mae_a = mean_absolute_error(y_test, pred_a)
rmse_a = mean_squared_error(y_test, pred_a) ** 0.5
r2_a = r2_score(y_test, pred_a)

print("Modelo A (básico)            -> MAE:", mae_a, " RMSE:", rmse_a, " R2:", r2_a)
print("Modelo B (Feature Engineering) -> MAE:", mae_b, " RMSE:", rmse_b, " R2:", r2_b)


Modelo A (básico)            -> MAE: 359804.2855792781  RMSE: 426638.58671856375  R2: 0.5338623188445003
Modelo B (Feature Engineering) -> MAE: 89889.43759816857  RMSE: 108444.96720760677  R2: 0.96988292774666


**Pregunta para el grupo:** ¿qué variables nuevas pudieron aportar más
información? (pista: revisa el efecto de `zona` y `tipo_vivienda` en el precio).

---
## Feature Importance con Random Forest
Usamos solo las columnas numéricas originales, como en la lámina
"Ejemplo rápido".

In [325]:
from sklearn.ensemble import RandomForestRegressor

X_train_num = X_train[numericas]
X_test_num = X_test[numericas]

modelo_rf = RandomForestRegressor(n_estimators=100, random_state=42)
modelo_rf.fit(X_train_num, y_train)

for nombre, importancia in zip(numericas, modelo_rf.feature_importances_):
    print(nombre, round(importancia, 4))


metros 0.6186
habitaciones 0.0147
banios 0.0207
antiguedad 0.1235
mes_publicacion 0.0502
dia_semana 0.0436
metros_por_habitacion 0.0816
banios_por_habitacion 0.0471


---
## Preguntas de salida (para cerrar la sección)

1. Con `KNN`, ¿qué transformación investigarías primero dado que
   `metros` e `ingreso` tienen escalas tan distintas?
2. `zona` no tiene orden natural. ¿Qué técnica de codificación usarías?
3. Si aplicas `StandardScaler.fit(X)` antes de separar train/test,
   ¿qué problema se introduce?